# FASTA File Parser

This notebook reads a FASTA file and extracts the sequence data into a pandas DataFrame for further analysis.

The FASTA file contains protein sequences with headers that include:
- Protein ID (e.g., tr|A0A024R324|A0A024R324_HUMAN)
- Protein description
- Organism (OS)
- Organism identifier (OX)
- Gene name (GN)
- Protein existence (PE)
- Sequence version (SV)

In [2]:
import pandas as pd
import re
from pathlib import Path
import os

In [3]:
def parse_fasta_header(header):
    """
    Parse a FASTA header and extract relevant metadata.
    
    Args:
        header (str): FASTA header line (without the '>')
        
    Returns:
        dict: Dictionary containing parsed metadata
    """
    # Initialize result dictionary
    result = {
        'protein_id': '',
        'protein_name': '',
        'organism': '',
        'organism_id': '',
        'gene_name': '',
        'protein_existence': '',
        'sequence_version': '',
        'full_header': header
    }
    
    # Extract protein ID (first part before space)
    parts = header.split(' ', 1)
    if len(parts) > 0:
        result['protein_id'] = parts[0].split('|')[1] if '|' in parts[0] else parts[0]
    
    if len(parts) > 1:
        description = parts[1]
        
        # Extract protein name (everything before OS=)
        if 'OS=' in description:
            result['protein_name'] = description.split('OS=')[0].strip()
        else:
            result['protein_name'] = description.strip()
        
        # Extract organism (OS=...)
        os_match = re.search(r'OS=([^=]+?)(?:\s+[A-Z]{2}=|$)', description)
        if os_match:
            result['organism'] = os_match.group(1).strip()
        
        # Extract organism ID (OX=...)
        ox_match = re.search(r'OX=(\d+)', description)
        if ox_match:
            result['organism_id'] = ox_match.group(1)
        
        # Extract gene name (GN=...)
        gn_match = re.search(r'GN=([^=]+?)(?:\s+[A-Z]{2}=|$)', description)
        if gn_match:
            result['gene_name'] = gn_match.group(1).strip()
        
        # Extract protein existence (PE=...)
        pe_match = re.search(r'PE=(\d+)', description)
        if pe_match:
            result['protein_existence'] = pe_match.group(1)
        
        # Extract sequence version (SV=...)
        sv_match = re.search(r'SV=(\d+)', description)
        if sv_match:
            result['sequence_version'] = sv_match.group(1)
    
    return result

# Test the function with a sample header
test_header = "tr|A0A024R324|A0A024R324_HUMAN Transforming protein RhoA OS=Homo sapiens OX=9606 GN=RHOA PE=2 SV=1"
parsed = parse_fasta_header(test_header)
print("Test parsing result:")
for key, value in parsed.items():
    print(f"  {key}: {value}")

Test parsing result:
  protein_id: A0A024R324
  protein_name: Transforming protein RhoA
  organism: Homo sapiens
  organism_id: 9606
  gene_name: RHOA
  protein_existence: 2
  sequence_version: 1
  full_header: tr|A0A024R324|A0A024R324_HUMAN Transforming protein RhoA OS=Homo sapiens OX=9606 GN=RHOA PE=2 SV=1


In [4]:
def fasta_to_dataframe(fasta_file_path):
    """
    Read a FASTA file and convert it to a pandas DataFrame.
    
    Args:
        fasta_file_path (str): Path to the FASTA file
        
    Returns:
        pd.DataFrame: DataFrame containing sequence data and metadata
    """
    sequences = []
    current_header = ""
    current_sequence = ""
    
    try:
        with open(fasta_file_path, 'r') as file:
            for line in file:
                line = line.strip()
                
                if line.startswith('>'):
                    # Save previous sequence if exists
                    if current_header and current_sequence:
                        # Parse header
                        metadata = parse_fasta_header(current_header[1:])  # Remove '>'
                        metadata['sequence'] = current_sequence
                        metadata['sequence_length'] = len(current_sequence)
                        sequences.append(metadata)
                    
                    # Start new sequence
                    current_header = line
                    current_sequence = ""
                else:
                    # Add to current sequence
                    current_sequence += line
            
            # Don't forget the last sequence
            if current_header and current_sequence:
                metadata = parse_fasta_header(current_header[1:])
                metadata['sequence'] = current_sequence
                metadata['sequence_length'] = len(current_sequence)
                sequences.append(metadata)
    
    except FileNotFoundError:
        print(f"Error: File {fasta_file_path} not found!")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error reading file: {e}")
        return pd.DataFrame()
    
    # Create DataFrame
    df = pd.DataFrame(sequences)
    
    # Reorder columns for better readability
    column_order = ['protein_id', 'protein_name', 'organism', 'organism_id', 
                   'gene_name', 'protein_existence', 'sequence_version',
                   'sequence_length', 'sequence', 'full_header']
    
    # Only include columns that exist in the DataFrame
    available_columns = [col for col in column_order if col in df.columns]
    df = df[available_columns]
    
    return df

print("FASTA parsing function defined!")

FASTA parsing function defined!


## Load and Parse FASTA File

Read the FASTA file and convert it to a DataFrame.

In [5]:
# Specify the path to your FASTA file
fasta_file_path = "strep.fasta"
organism = fasta_file_path.split(".")[0]

# Check if file exists
if os.path.exists(fasta_file_path):
    print(f"File found: {fasta_file_path}")
    
    # Read the FASTA file and convert to DataFrame
    print("Reading FASTA file...")
    df = fasta_to_dataframe(fasta_file_path)
    
    print(f"Successfully loaded {len(df)} sequences!")
    print(f"DataFrame shape: {df.shape}")
else:
    print(f"File not found: {fasta_file_path}")
    print("Please check the file path and try again.")

File found: strep.fasta
Reading FASTA file...
Successfully loaded 917221 sequences!
DataFrame shape: (917221, 10)
Successfully loaded 917221 sequences!
DataFrame shape: (917221, 10)


In [6]:
# Display basic information about the DataFrame
print("DataFrame Info:")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\nData types:")
print(df.dtypes)
print("\nFirst few rows:")
df.head()

DataFrame Info:
Shape: (917221, 10)
Columns: ['protein_id', 'protein_name', 'organism', 'organism_id', 'gene_name', 'protein_existence', 'sequence_version', 'sequence_length', 'sequence', 'full_header']

Data types:
protein_id           object
protein_name         object
organism             object
organism_id          object
gene_name            object
protein_existence    object
sequence_version     object
sequence_length       int64
sequence             object
full_header          object
dtype: object

First few rows:


,protein_id,protein_name,organism,organism_id,gene_name,protein_existence,sequence_version,sequence_length,sequence,full_header
0,K7QVW7,Spiro-conjugate synthase,Streptomyces rugosporus,295838,pyrI4,1,2,184,MTTPQIDERAMEAGAAALQETIVDPGPLDVTALAVAAALAAGLHSA...,sp|K7QVW7|PYRI4_STRRG Spiro-conjugate synthase...
1,O52646,Aklanonic acid methyl ester cyclase AcmA,Streptomyces galilaeus,33899,acma,1,1,144,MSEQIAAVRRMVEAYNTGKTDDVADYIHPEYMNPGTLEFTSLRGPE...,sp|O52646|DNRD_STRGJ Aklanonic acid methyl est...
2,O54259,Deoxynogalonate monooxygenase,Streptomyces nogalater,38314,snoaB,1,1,118,MPTRVNDGVDADEVTFVNRFTVHGGPAEFESVFARTAAFFARQPGF...,sp|O54259|SNOAB_STRNO Deoxynogalonate monooxyg...
3,O88038,Lanthionine-containing peptide SapB precursor ...,Streptomyces coelicolor (strain ATCC BAA-471 /...,100226,ramS,1,1,42,MNLFDLQSMETPKEEAMGDVETGSRASLLLCGDSSLSITTCN,sp|O88038|LANSB_STRCO Lanthionine-containing p...
4,P01006,Subtilisin inhibitor,Streptomyces albogriseolus,1887,ssi,1,2,144,MRNTGAGPSPSVSRPPPSAAPLSGAALAAPGDAPSALYAPSALVLT...,sp|P01006|SSI_STRAO Subtilisin inhibitor OS=St...


## Save DataFrame to CSV

In [7]:
# Define output path
output_path = f"./info_{organism}.csv"

# Save to CSV
df.to_csv(output_path, index=False)
print(f"DataFrame saved to: {output_path}")
print(f"Saved {len(df)} sequences with {len(df.columns)} columns")

DataFrame saved to: ./info_strep.csv
Saved 917221 sequences with 10 columns
